# GraphToken


### 1. Dependencies & Random Seeds


In [ ]:
import os
import random
import re
from dataclasses import dataclass
from typing import Any

import torch
import torch.nn as nn
from datasets import load_dataset

# PyTorch Geometric
from torch_geometric.data import Batch as PyGBatch
from torch_geometric.data import Data as PyGData
from torch_geometric.nn import GCNConv, global_mean_pool
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Expose only one GPU (must be set at the very top)

SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)

### 2. Constants and Model/Tokenizer Setup


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  # Pretrained LLM (frozen)
PROMPT_TOKENS = 16  # Length of the graph-derived continuous prompt (tune as needed)
MAX_LENGTH = 2048  # Maximum sequence length per sample (text side)
LR = 1e-4  # Learning rate for GNN/FC (adjust around 3e-5 to 1e-4 if needed)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### 3. Load Dataset (Hugging Face Hub)

Load the zero-shot split from the GraphQA edge_count subset.


In [ ]:
BASE = "baharef/GraphQA"
DATA_FILES = {
    "train": "edge_count/edge_count_zero_shot_train.json",
    "validation": "edge_count/edge_count_zero_shot_validation.json",
    "test": "edge_count/edge_count_zero_shot_test.json",
}

ds = load_dataset(BASE, data_files=DATA_FILES)
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['algorithm', 'answer', 'nedges', 'nnodes', 'question', 'task_description', 'text_encoding'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['algorithm', 'answer', 'nedges', 'nnodes', 'question', 'task_description', 'text_encoding'],
        num_rows: 500
    })
    test: Dataset({
        features: ['algorithm', 'answer', 'nedges', 'nnodes', 'question', 'task_description', 'text_encoding'],
        num_rows: 500
    })
})
{'algorithm': 'er', 'answer': ' 14.', 'nedges': '14', 'nnodes': '8', 'question': 'In an undirected graph, (i,j) means that node i and node j are connected with an undirected edge. G describes a graph among nodes 0, 1, 2, 3, 4, 5, 6, and 7.\nThe edges in G are: (0, 1) (0, 5) (0, 6) (0, 7) (1, 2) (1, 4) (1, 5) (1, 6) (1, 7) (2, 3) (2, 4) (3, 7) (4, 5) (5, 7).\nQ: How many edges are in this graph?\nA: ', 'task_description': 'Q: How many edges are in this graph?\nA: ', 'text_encoding': 'adja

In [ ]:
# ★ Define this before creating the trainer
from torch.utils.data import Dataset as TorchDataset


class GraphQATorchDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[int(idx)]
        # Pass only the fields required by the collator (add others if needed)
        return {
            "question": ex["question"],
            "answer": ex["answer"],
        }


train_torch = GraphQATorchDataset(ds["train"])
val_torch = GraphQATorchDataset(ds["validation"])
test_torch = GraphQATorchDataset(ds["test"])

### 4. GraphToken-Style Architecture

- GraphEncoder: 2-layer GCN → global mean pooling → FC (hidden → d_model × P)
- GraphPromptLM: Freeze the LLM, concatenate prompt_embeds (B,P,d) before the text embeddings and run the forward pass via inputs_embeds. Use the LLM causal LM loss and prepend labels=-100 for the graph-token portion.


In [ ]:
class GraphEncoder(nn.Module):
    """Two-layer GCN + mean pool + FC to generate continuous soft prompts shaped (B, P, d_model)"""

    def __init__(self, in_dim=1, hidden=256, d_model=1536, prompt_tokens=PROMPT_TOKENS):
        super().__init__()
        self.gcn1 = GCNConv(in_dim, hidden)
        self.gcn2 = GCNConv(hidden, hidden)
        self.proj = nn.Linear(hidden, d_model * prompt_tokens)
        self.prompt_tokens = prompt_tokens
        self.d_model = d_model

    def forward(self, pyg_batch: PyGBatch):
        x = self.gcn1(pyg_batch.x, pyg_batch.edge_index).relu()
        x = self.gcn2(x, pyg_batch.edge_index).relu()
        g = global_mean_pool(x, pyg_batch.batch)  # (B, hidden)
        out = self.proj(g).view(-1, self.prompt_tokens, self.d_model)  # (B, P, d)
        return out


class GraphPromptLM(nn.Module):
    def __init__(self, model_id=MODEL_ID, prompt_tokens=PROMPT_TOKENS):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_id)
        for p in self.llm.parameters():
            p.requires_grad = False
        self.config = self.llm.config  # ★ Added: expose config referenced by TRL

        d_model = self.llm.get_input_embeddings().embedding_dim
        self.graph_encoder = GraphEncoder(d_model=d_model, prompt_tokens=prompt_tokens)
        self.embed_tokens = self.llm.get_input_embeddings()
        self.prompt_tokens = prompt_tokens

    def forward(self, input_ids=None, attention_mask=None, labels=None, graphs=None):
        # Text-side embeddings (allows retrieving the replica device)
        tok_embeds = self.embed_tokens(input_ids)  # (B_local, T, d)
        device = tok_embeds.device

        # Expect graphs as List[PyGData] per replica (DataParallel slices them)
        # Normalize to a list for compatibility even if Batch/single Data arrives
        if isinstance(graphs, list):
            data_list = [g.to(device) for g in graphs]
        elif hasattr(graphs, "to_data_list"):  # PyG Batch
            data_list = [g.to(device) for g in graphs.to_data_list()]
        else:  # Single Data
            data_list = [graphs.to(device)]

        # Build a replica-specific Batch (ensures B_local == len(data_list))
        graphs_local = PyGBatch.from_data_list(data_list)

        # Graphs -> continuous soft prompts
        prompt_embeds = self.graph_encoder(graphs_local)  # (B_local, P, d)

        # Concatenate and prepend labels filled with -100
        inputs_embeds = torch.cat([prompt_embeds, tok_embeds], dim=1)
        if labels is not None:
            pad = torch.full((labels.size(0), prompt_embeds.size(1)), -100, dtype=labels.dtype, device=labels.device)
            labels = torch.cat([pad, labels], dim=1)

        return self.llm(inputs_embeds=inputs_embeds, attention_mask=None, labels=labels)

### 5. Graph Extraction Utilities

Extract (i,j) edges from the GraphQA problem statements.


In [ ]:
edge_pat = re.compile(r"\((\d+),\s*(\d+)\)")


def parse_edges_from_question(q: str):
    edges = [(int(a), int(b)) for (a, b) in edge_pat.findall(q)]
    n_nodes = max((max(u, v) for u, v in edges), default=-1) + 1
    return edges, max(n_nodes, 1)

### 6. Custom Collator Performing Completion-Only Loss

- Use `apply_chat_template` to tokenize the prompt-only and prompt+assistant responses separately
- Start from `labels = input_ids.clone()` and set the leading prompt_len to -100 (loss only on assistant tokens)
- Pack the graphs into a PyG Batch and return them as `graphs`


In [ ]:
@dataclass
class GraphCollator:
    tokenizer: Any
    max_length: int = MAX_LENGTH
    graph_node_feat: float = 1.0  # Initial node feature (constant 1)

    def __call__(self, batch):
        # --- 1) Normalize batch structure ---
        if batch is None:
            raise ValueError("GraphCollator: received None batch")
        if isinstance(batch, dict):  # Handle the case where a single sample arrives as a dict
            batch = [batch]
        if len(batch) == 0:
            raise ValueError("GraphCollator: received empty batch")

        features, graphs = [], []

        for ex in batch:
            if "question" not in ex or "answer" not in ex:
                raise KeyError(
                    f"GraphCollator: missing keys; got {list(ex.keys())}. "
                    "Ensure TrainingArguments(remove_unused_columns=False)."
                )

            q, a = str(ex["question"]).strip(), str(ex["answer"]).strip()

            # --- 2) Completion-only: obtain prompt length ---
            prompt_only = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": q}],
                tokenize=False,
                add_generation_prompt=True,
            )
            enc_prompt = self.tokenizer(prompt_only, return_tensors="pt", add_special_tokens=False)
            prompt_len = enc_prompt["input_ids"].size(1)

            # --- 3) Tokenize the full sequence (user + assistant) ---
            full_text = self.tokenizer.apply_chat_template(
                [{"role": "user", "content": q}, {"role": "assistant", "content": a}],
                tokenize=False,
                add_generation_prompt=False,
            )
            enc_full = self.tokenizer(
                full_text,
                return_tensors="pt",
                add_special_tokens=False,
                truncation=True,
                max_length=self.max_length,
            )
            input_ids = enc_full["input_ids"][0]
            attention_mask = enc_full["attention_mask"][0]

            labels = input_ids.clone()
            labels[: min(prompt_len, labels.size(0))] = -100  # completion-only

            # ★ Key point: convert to list[int] before passing to pad
            features.append(
                {
                    "input_ids": input_ids.tolist(),
                    "attention_mask": attention_mask.tolist(),
                    "labels": labels.tolist(),
                }
            )

            # --- 4) Extract graphs → PyG Data ---
            edges, n = parse_edges_from_question(q)
            n = max(int(n), 1)
            x = torch.ones((n, 1), dtype=torch.float32) * self.graph_node_feat
            if len(edges) > 0:
                edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            else:
                edge_index = torch.empty((2, 0), dtype=torch.long)
            graphs.append(PyGData(x=x, edge_index=edge_index))

        # --- 5) Pad the text side (delegate only input_ids/attention_mask to tokenizer.pad) ---
        inputs_for_pad = [{"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]} for f in features]
        padded_inputs = self.tokenizer.pad(
            inputs_for_pad,
            padding=True,  # or padding="max_length" + max_length=self.max_length
            return_tensors="pt",
        )

        # --- 5') Manually pad labels (right-pad with -100) ---
        max_len = padded_inputs["input_ids"].size(1)
        padded_labels = []
        for f in features:
            lab = f["labels"]
            if len(lab) > max_len:
                # Just in case (enc_full truncation usually prevents this)
                lab = lab[:max_len]
            if len(lab) < max_len:
                lab = lab + ([-100] * (max_len - len(lab)))
            padded_labels.append(lab)
        padded_labels = torch.tensor(padded_labels, dtype=torch.long)

        # --- 6) Batch the graph side ---
        if len(graphs) == 0:
            graphs = [
                PyGData(x=torch.ones((1, 1), dtype=torch.float32), edge_index=torch.empty((2, 0), dtype=torch.long))
            ]
        # pyg_batch = PyGBatch.from_data_list(graphs)

        return {
            "input_ids": padded_inputs["input_ids"],
            "attention_mask": padded_inputs["attention_mask"],
            "labels": padded_labels,
            "graphs": graphs,  # ✅ Return as a List[PyGData]
        }


collator = GraphCollator(tokenizer=tokenizer, max_length=MAX_LENGTH)

In [17]:
from torch.utils.data import DataLoader

dl = DataLoader(train_torch, batch_size=1, shuffle=False, collate_fn=collator)
first_batch = next(iter(dl))
for k, v in first_batch.items():
    print(k, type(v), getattr(v, "shape", None))
# Check here that "graphs" is a PyG Batch and that input_ids/labels have the expected shapes

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


input_ids <class 'torch.Tensor'> torch.Size([1, 175])
attention_mask <class 'torch.Tensor'> torch.Size([1, 175])
labels <class 'torch.Tensor'> torch.Size([1, 175])
graphs <class 'list'> None


### 7. Trainer Setup & Training

SFTTrainer in the latest TRL no longer accepts `tokenizer=`; pass `processing_class=tokenizer` instead.


In [ ]:
from transformers import Trainer, TrainingArguments

# model = GraphPromptLM(MODEL_ID, PROMPT_TOKENS)
model = GraphPromptLM(MODEL_ID, PROMPT_TOKENS).to("cuda")


training_args = TrainingArguments(
    output_dir="glm-graphprompt-qwen3-4b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    num_train_epochs=1,
    logging_steps=20,
    save_steps=1000,
    bf16=True,
    fp16=False,
    max_grad_norm=1.0,
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_torch,  # Wrapped Torch dataset
    eval_dataset=val_torch,
    data_collator=collator,  # Custom collator (completion-only + graph)
)

trainer.train()

Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.41it/s]


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/torch/nn/parallel/parallel_apply.py", line 84, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_165293/1680969896.py", line 54, in forward
    inputs_embeds = torch.cat([prompt_embeds, tok_embeds], dim=1)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Sizes of tensors must match except i

### 8. Inference Helper & Quick Evaluation (zero_shot_test)

- During inference, use inputs_embeds to concatenate the graph-derived prompt with the user prompt
- Approximate accuracy via simple integer comparison (edge_count answers are integers)


In [ ]:
num_pat = re.compile(r"(-?\d+)")


def generate_answer(model: GraphPromptLM, question: str, max_new_tokens: int = 32):
    edges, n = parse_edges_from_question(question)
    x = torch.ones((n, 1), dtype=torch.float32)
    edge_index = (
        torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    )
    pyg_batch = PyGBatch.from_data_list([PyGData(x=x, edge_index=edge_index)])

    messages = [{"role": "user", "content": question.strip()}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tok = tokenizer(prompt, return_tensors="pt")

    model.eval()
    with torch.no_grad():
        tok_embeds = model.embed_tokens(tok["input_ids"])  # (1, T, d)
        gprompt = model.graph_encoder(pyg_batch)  # (1, P, d)
        inputs_embeds = torch.cat([gprompt, tok_embeds], dim=1)  # (1, P+T, d)
        out_ids = model.llm.generate(inputs_embeds=inputs_embeds, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)


# Quick accuracy (compare the first integers)
correct = 0
for ex in ds["test"]:
    pred = generate_answer(model, ex["question"])
    tp = num_pat.findall(pred)
    ta = num_pat.findall(ex["answer"])
    if tp and ta and tp[0] == ta[0]:
        correct += 1
acc = correct / len(ds["test"]) if len(ds["test"]) else float("nan")
print(f"Test Accuracy: {acc:.3f}")

### 9. Saving (trained GNN + FC only)


In [ ]:
# Because the LLM remains frozen, this example saves only the graph_encoder weights
os.makedirs("glm-graphprompt-qwen3-4b/artifacts", exist_ok=True)
torch.save(model.graph_encoder.state_dict(), "glm-graphprompt-qwen3-4b/artifacts/graph_encoder.pt")